In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv') # (Check spelling if your file has an extra 'e' like delieveries.csv)
ipl_combined = pd.read_csv('../data/raw/IPL.csv')

print(f"Matches shape: {matches.shape[0]} rows")
print(f"Deliveries shape: {deliveries.shape[0]} rows")
print(f"IPL Combined shape: {ipl_combined.shape[0]} rows")

Matches shape: 1095 rows
Deliveries shape: 260920 rows
IPL Combined shape: 278205 rows


In [2]:
# 1. Total Matches
print(f"Total Matches Played: {len(matches)}")

# 2. Most Matches Won by a Team
print("\n--- Top 5 Teams by Matches Won ---")
print(matches['winner'].value_counts().head(5))

# 3. Toss Impact: Does winning the toss mean winning the match?
toss_win_match_win = (matches['toss_winner'] == matches['winner']).mean() * 100
print(f"\n--- Toss Impact ---")
print(f"Percentage of matches won by the toss winner: {toss_win_match_win:.2f}%")

# 4. Toss Decision Trend (Bat vs Field over the years)
print("\n--- Toss Decisions Over Time ---")
toss_trend = matches.groupby('season')['toss_decision'].value_counts(normalize=True).unstack().fillna(0) * 100
# Showing just the last 5 seasons for a quick look
display(toss_trend.tail(5).round(1))

Total Matches Played: 1095

--- Top 5 Teams by Matches Won ---
winner
Mumbai Indians                 144
Chennai Super Kings            138
Kolkata Knight Riders          131
Royal Challengers Bangalore    116
Rajasthan Royals               112
Name: count, dtype: int64

--- Toss Impact ---
Percentage of matches won by the toss winner: 50.59%

--- Toss Decisions Over Time ---


toss_decision,bat,field
season,,
2020/21,45.0,55.0
2021,26.7,73.3
2022,20.3,79.7
2023,28.4,71.6
2024,26.8,73.2


In [3]:
# 1. Total Balls Bowled
print(f"Total Balls Bowled: {len(deliveries)}")

# 2. Top 5 Run Scorers of All Time
print("\n--- Top 5 Run Scorers ---")
top_batters = deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(5)
print(top_batters)

# 3. Top 5 Wicket Takers (Counting all dismissals where a wicket fell)
print("\n--- Top 5 Wicket Takers ---")
top_bowlers = deliveries[deliveries['is_wicket'] == 1].groupby('bowler').size().sort_values(ascending=False).head(5)
print(top_bowlers)

# 4. Impact of Extras (Wides, No Balls, etc.)
total_runs = deliveries['total_runs'].sum()
total_extras = deliveries['extra_runs'].sum()
extras_percentage = (total_extras / total_runs) * 100
print(f"\n--- Extras Impact ---")
print(f"Percentage of total runs that come from extras: {extras_percentage:.2f}%")

Total Balls Bowled: 260920

--- Top 5 Run Scorers ---
batter
V Kohli      8014
S Dhawan     6769
RG Sharma    6630
DA Warner    6567
SK Raina     5536
Name: batsman_runs, dtype: int64

--- Top 5 Wicket Takers ---
bowler
YS Chahal    213
DJ Bravo     207
PP Chawla    201
SP Narine    200
R Ashwin     198
dtype: int64

--- Extras Impact ---
Percentage of total runs that come from extras: 5.09%


In [4]:
# Dictionary to update old franchise names to their modern names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru' # Handling the recent 2024 name change
}

# Update in the matches dataset
matches['team1'] = matches['team1'].replace(team_mapping)
matches['team2'] = matches['team2'].replace(team_mapping)
matches['winner'] = matches['winner'].replace(team_mapping)
matches['toss_winner'] = matches['toss_winner'].replace(team_mapping)

# Update in the deliveries dataset
deliveries['batting_team'] = deliveries['batting_team'].replace(team_mapping)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_mapping)

print("--- Active Teams After Cleaning ---")
print(matches['team1'].unique())

--- Active Teams After Cleaning ---
['Royal Challengers Bengaluru' 'Punjab Kings' 'Delhi Capitals'
 'Mumbai Indians' 'Kolkata Knight Riders' 'Rajasthan Royals'
 'Sunrisers Hyderabad' 'Chennai Super Kings' 'Kochi Tuskers Kerala'
 'Pune Warriors' 'Gujarat Lions' 'Rising Pune Supergiant'
 'Lucknow Super Giants' 'Gujarat Titans']


In [5]:
# 1. Drop matches that had no result (rain washouts)
matches.dropna(subset=['winner'], inplace=True)

# 2. Create our Target Variable: 1 if team1 wins, 0 if team2 wins
matches['team1_win'] = (matches['team1'] == matches['winner']).astype(int)

# 3. Create a clean dataframe for our Pre-Match Engine
pre_match_df = matches[['id', 'season', 'city', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'team1_win']].copy()

print(f"Ready for feature engineering! Total valid matches: {len(pre_match_df)}")
display(pre_match_df.head(3))

Ready for feature engineering! Total valid matches: 1090


,id,season,city,venue,team1,team2,toss_winner,toss_decision,team1_win
0,335982,2007/08,Bangalore,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,0
1,335983,2007/08,Chandigarh,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,0
2,335984,2007/08,Delhi,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,1


In [6]:
# 1. Feature: Did Team 1 win the toss? (1 = Yes, 0 = No)
pre_match_df['team1_toss_win'] = (pre_match_df['team1'] == pre_match_df['toss_winner']).astype(int)

# 2. Feature: Is Team 1 batting first? (1 = Yes, 0 = No)
# Logic: If Team 1 won toss & chose bat OR if Team 2 won toss & chose field
pre_match_df['team1_batting'] = (
    ((pre_match_df['team1_toss_win'] == 1) & (pre_match_df['toss_decision'] == 'bat')) |
    ((pre_match_df['team1_toss_win'] == 0) & (pre_match_df['toss_decision'] == 'field'))
).astype(int)

# 3. Drop the old text columns we just converted
pre_match_df = pre_match_df.drop(columns=['toss_winner', 'toss_decision'])

print("--- New Engineered Features ---")
display(pre_match_df.head(3))

--- New Engineered Features ---


,id,season,city,venue,team1,team2,team1_win,team1_toss_win,team1_batting
0,335982,2007/08,Bangalore,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,0,1,0
1,335983,2007/08,Chandigarh,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,0,0,0
2,335984,2007/08,Delhi,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,1,0,0


In [7]:
# 1. Keep only the columns necessary for the prediction model
model_df = pre_match_df[['team1', 'team2', 'venue', 'team1_toss_win', 'team1_batting', 'team1_win']].copy()

# 2. Apply One-Hot Encoding to convert text into 1s and 0s
model_df = pd.get_dummies(model_df, columns=['team1', 'team2', 'venue'])

# 3. Ensure everything is an integer (turns True/False into 1/0)
model_df = model_df.astype(int)

print(f"Final model shape: {model_df.shape[0]} matches, {model_df.shape[1]} features")
display(model_df.head(3))

Final model shape: 1090 matches, 89 features


,team1_toss_win,team1_batting,team1_win,team1_Chennai Super Kings,team1_Delhi Capitals,team1_Gujarat Lions,team1_Gujarat Titans,team1_Kochi Tuskers Kerala,team1_Kolkata Knight Riders,team1_Lucknow Super Giants,...,venue_Shaheed Veer Narayan Singh International Stadium,venue_Sharjah Cricket Stadium,venue_Sheikh Zayed Stadium,venue_St George's Park,venue_Subrata Roy Sahara Stadium,venue_SuperSport Park,"venue_Vidarbha Cricket Association Stadium, Jamtha",venue_Wankhede Stadium,"venue_Wankhede Stadium, Mumbai","venue_Zayed Cricket Stadium, Abu Dhabi"
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Separate Features (X) and Target (y)
X = model_df.drop('team1_win', axis=1)
y = model_df['team1_win']

# 2. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Baseline Model: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

# 4. Train Advanced Model: Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Logistic Regression Accuracy: {lr_accuracy * 100:.2f}%")
print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

Logistic Regression Accuracy: 57.34%
Random Forest Accuracy: 54.13%


In [9]:
# 1. Calculate total matches and total wins for every team
team_matches = matches['team1'].value_counts() + matches['team2'].value_counts()
team_wins = matches['winner'].value_counts()

# 2. Calculate the historical win rate
team_win_rate = (team_wins / team_matches).fillna(0)

# 3. Map these win rates into our dataframe
pre_match_df['team1_win_rate'] = pre_match_df['team1'].map(team_win_rate)
pre_match_df['team2_win_rate'] = pre_match_df['team2'].map(team_win_rate)

# 4. Create the powerful new feature: Strength Difference
# A positive number means Team 1 is stronger. A negative number means Team 2 is stronger.
pre_match_df['team_strength_diff'] = pre_match_df['team1_win_rate'] - pre_match_df['team2_win_rate']

print("--- New Team Strength Features ---")
display(pre_match_df[['team1', 'team2', 'team1_win_rate', 'team2_win_rate', 'team_strength_diff']].head(5))

--- New Team Strength Features ---


,team1,team2,team1_win_rate,team2_win_rate,team_strength_diff
0,Royal Challengers Bengaluru,Kolkata Knight Riders,0.488095,0.521912,-0.033817
1,Punjab Kings,Chennai Super Kings,0.455285,0.582278,-0.126994
2,Delhi Capitals,Rajasthan Royals,0.460000,0.511416,-0.051416
3,Mumbai Indians,Royal Challengers Bengaluru,0.551724,0.488095,0.063629
4,Kolkata Knight Riders,Sunrisers Hyderabad,0.521912,0.455253,0.066659


In [10]:
# 1. Grab our updated features
model_df_v2 = pre_match_df[['team1', 'team2', 'venue', 'team1_toss_win', 'team1_batting', 'team_strength_diff', 'team1_win']].copy()

# 2. One-Hot Encode text columns again
model_df_v2 = pd.get_dummies(model_df_v2, columns=['team1', 'team2', 'venue'])

# 3. Convert booleans to 1s and 0s safely without destroying our decimal numbers
for col in model_df_v2.columns:
    if model_df_v2[col].dtype == bool:
        model_df_v2[col] = model_df_v2[col].astype(int)

# 4. Train-Test Split
X = model_df_v2.drop('team1_win', axis=1)
y = model_df_v2['team1_win']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train and Predict with the new data
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_accuracy_v2 = accuracy_score(y_test, lr_model.predict(X_test))

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_accuracy_v2 = accuracy_score(y_test, rf_model.predict(X_test))

print(f"NEW Logistic Regression Accuracy: {lr_accuracy_v2 * 100:.2f}%")
print(f"NEW Random Forest Accuracy: {rf_accuracy_v2 * 100:.2f}%")

NEW Logistic Regression Accuracy: 57.80%
NEW Random Forest Accuracy: 51.83%


In [11]:
# 1. Create a clean list of every time a team played at a venue
team_venues = pd.concat([
    matches[['venue', 'team1', 'winner']].rename(columns={'team1': 'team'}),
    matches[['venue', 'team2', 'winner']].rename(columns={'team2': 'team'})
])

# 2. Mark 1 if they won, 0 if they lost
team_venues['won'] = (team_venues['team'] == team_venues['winner']).astype(int)

# 3. Calculate historical win rate per team PER VENUE
venue_stats = team_venues.groupby(['venue', 'team'])['won'].mean().reset_index()
venue_stats.rename(columns={'won': 'venue_win_rate'}, inplace=True)

# 4. Map Team 1's Venue Win Rate
pre_match_df = pd.merge(pre_match_df, venue_stats, left_on=['venue', 'team1'], right_on=['venue', 'team'], how='left')
pre_match_df.rename(columns={'venue_win_rate': 'team1_venue_win_rate'}, inplace=True)
pre_match_df.drop('team', axis=1, inplace=True)

# 5. Map Team 2's Venue Win Rate
pre_match_df = pd.merge(pre_match_df, venue_stats, left_on=['venue', 'team2'], right_on=['venue', 'team'], how='left')
pre_match_df.rename(columns={'venue_win_rate': 'team2_venue_win_rate'}, inplace=True)
pre_match_df.drop('team', axis=1, inplace=True)

# 6. Fill any missing data with 0 (if a team never played there before) and calculate the difference
pre_match_df.fillna(0, inplace=True)
pre_match_df['venue_dna_diff'] = pre_match_df['team1_venue_win_rate'] - pre_match_df['team2_venue_win_rate']

print("--- Venue DNA Score Added ---")
display(pre_match_df[['venue', 'team1', 'team2', 'team1_venue_win_rate', 'team2_venue_win_rate', 'venue_dna_diff']].head(5))

--- Venue DNA Score Added ---


,venue,team1,team2,team1_venue_win_rate,team2_venue_win_rate,venue_dna_diff
0,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,0.491525,0.545455,-0.053929
1,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,0.514286,0.750000,-0.235714
2,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,0.418182,0.571429,-0.153247
3,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,0.626866,0.300000,0.326866
4,Eden Gardens,Kolkata Knight Riders,Sunrisers Hyderabad,0.608108,0.181818,0.426290


In [12]:
from xgboost import XGBClassifier

# 1. Grab all updated features
model_df_v3 = pre_match_df[['team1', 'team2', 'venue', 'team1_toss_win', 'team1_batting', 
                            'team_strength_diff', 'venue_dna_diff', 'team1_win']].copy()

# 2. One-Hot Encode
model_df_v3 = pd.get_dummies(model_df_v3, columns=['team1', 'team2', 'venue'])

# 3. Convert booleans to integers securely
for col in model_df_v3.columns:
    if model_df_v3[col].dtype == bool:
        model_df_v3[col] = model_df_v3[col].astype(int)

# 4. Train-Test Split
X = model_df_v3.drop('team1_win', axis=1)
y = model_df_v3['team1_win']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train all 3 Models
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_accuracy_v3 = accuracy_score(y_test, lr_model.predict(X_test))

# Random Forest (Added max_depth=10 to stop it from overfitting!)
rf_model = RandomForestClassifier(random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
rf_accuracy_v3 = accuracy_score(y_test, rf_model.predict(X_test))

# XGBoost (The Heavyweight)
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_accuracy = accuracy_score(y_test, xgb_model.predict(X_test))

print(f"V3 Logistic Regression Accuracy: {lr_accuracy_v3 * 100:.2f}%")
print(f"V3 Random Forest Accuracy (Tuned): {rf_accuracy_v3 * 100:.2f}%")
print(f"XGBoost Accuracy: {xgb_accuracy * 100:.2f}%")

V3 Logistic Regression Accuracy: 69.72%
V3 Random Forest Accuracy (Tuned): 70.64%
XGBoost Accuracy: 66.06%


In [13]:
import joblib
import os

# 1. Ensure the models directory exists (going up one level from 'notebooks')
model_path = '../models/'
if not os.path.exists(model_path):
    os.makedirs(model_path)

# 2. Save the winning Random Forest model
joblib.dump(rf_model, os.path.join(model_path, 'pre_match_rf_model.pkl'))

# 3. Save the exact list of columns (features) 
# This is CRITICAL so the Streamlit app knows the exact order of 1s and 0s later
model_features = list(X_train.columns)
joblib.dump(model_features, os.path.join(model_path, 'pre_match_features.pkl'))

print("✅ Success! Phase 2 is officially complete.")
print(f"Files saved in /models: \n1. pre_match_rf_model.pkl \n2. pre_match_features.pkl")

✅ Success! Phase 2 is officially complete.
Files saved in /models: 
1. pre_match_rf_model.pkl 
2. pre_match_features.pkl


In [14]:
import numpy as np

# 1. Calculate cumulative score per inning
deliveries['current_score'] = deliveries.groupby(['match_id', 'inning'])['total_runs'].cumsum()

# 2. Get the target and city from matches.csv
# We need 'id' to match, 'target' to calculate runs left, and 'city' for the model
match_info = matches[['id', 'city', 'winner', 'team2']].copy()
match_info.rename(columns={'id': 'match_id'}, inplace=True)

# 3. Calculate target (1st innings total + 1)
first_innings_total = deliveries.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()
target_df = first_innings_total[first_innings_total['inning'] == 1]
target_df['target'] = target_df['total_runs'] + 1
target_df = target_df[['match_id', 'target']]

# 4. Merge everything into one Live Dataframe
live_df = deliveries.merge(target_df, on='match_id')
live_df = live_df.merge(match_info, on='match_id')

# 5. Filter for 2nd Inning (The Chase)
live_df = live_df[live_df['inning'] == 2]

# 6. Calculate real-time features
live_df['runs_left'] = live_df['target'] - live_df['current_score']
live_df['balls_left'] = 126 - (live_df['over'] * 6 + live_df['ball']) # Standard IPL ball count logic
live_df['wickets_left'] = 10 - live_df.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

# 7. Calculate CRR and RRR
# We add 0.001 to avoid division by zero errors
live_df['crr'] = (live_df['current_score'] * 6) / (120 - live_df['balls_left'] + 0.001)
live_df['rrr'] = (live_df['runs_left'] * 6) / (live_df['balls_left'] + 0.001)

# 8. Define the Result (1 if batting team won, 0 if they lost)
live_df['result'] = (live_df['team2'] == live_df['winner']).astype(int)

# 9. Final Selection
final_live_df = live_df[['batting_team', 'bowling_team', 'city', 'runs_left', 
                         'balls_left', 'wickets_left', 'target', 'crr', 'rrr', 'result']]

# Clean up: Remove any rows with negative balls or missing cities
final_live_df = final_live_df[final_live_df['balls_left'] > 0]
final_live_df.dropna(inplace=True)

print(f"Phase 3 Data Ready! Total Match States: {final_live_df.shape[0]}")
display(final_live_df.head())

Phase 3 Data Ready! Total Match States: 119702


,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,target,crr,rrr,result
124,Royal Challengers Bengaluru,Kolkata Knight Riders,Bangalore,222,125,10,223,-1.200240,10.655915,1
125,Royal Challengers Bengaluru,Kolkata Knight Riders,Bangalore,221,124,10,223,-3.000750,10.693462,1
126,Royal Challengers Bengaluru,Kolkata Knight Riders,Bangalore,221,123,10,223,-4.001334,10.780400,1
127,Royal Challengers Bengaluru,Kolkata Knight Riders,Bangalore,220,122,10,223,-9.004502,10.819583,1
128,Royal Challengers Bengaluru,Kolkata Knight Riders,Bangalore,219,121,10,223,-24.024024,10.859414,1


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Define Features (X) and Target (y)
X = final_live_df.drop('result', axis=1)
y = final_live_df['result']

# 2. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create a Pipeline
# This handles One-Hot Encoding for teams/city AND the model in one go
step1 = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

step2 = LogisticRegression(solver='liblinear')

pipe = Pipeline([
    ('step1', step1),
    ('step2', step2)
])

# 4. Train the Model
pipe.fit(X_train, y_train)

# 5. Check Accuracy
y_pred = pipe.predict(X_test)
print(f"Live Match Predictor Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Live Match Predictor Accuracy: 64.58%


In [16]:
# 1. Load the advanced IPL.csv data
ipl_df = pd.read_csv('../data/raw/IPL.csv')

# 2. Calculate "Recent Momentum" (Last 18 valid balls)
# We use a rolling sum to see how many runs/wickets happened recently
ipl_df['recent_runs'] = ipl_df.groupby(['match_id', 'innings'])['runs_batter'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())
ipl_df['recent_wickets'] = ipl_df.groupby(['match_id', 'innings'])['bowler_wicket'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())

# 3. Calculate "Partnership Total"
# This is already in your IPL.csv! We just need to make sure it's used.
ipl_df['current_partnership'] = ipl_df['batting_partners'] 

# 4. Filter for 2nd Innings and Merge with our previous Match State logic
# (I'm combining the logic here to make it one clean run)
live_pro_df = ipl_df[ipl_df['innings'] == 2].copy()

# Add the basic live stats we made earlier
live_pro_df['current_score'] = live_pro_df.groupby(['match_id', 'innings'])['runs_total'].cumsum()
live_pro_df['runs_left'] = live_pro_df['runs_target'] - live_pro_df['current_score']
live_pro_df['balls_left'] = 126 - (live_pro_df['over'] * 6 + live_pro_df['ball'])
live_pro_df['wickets_left'] = 10 - live_pro_df.groupby(['match_id', 'innings'])['bowler_wicket'].cumsum()

# 5. THE GAME CHANGER: The Pressure Index Formula
# Higher RRR + Recent Wickets = High Pressure
live_pro_df['rrr'] = (live_pro_df['runs_left'] * 6) / (live_pro_df['balls_left'] + 0.001)
live_pro_df['pressure_index'] = (live_pro_df['rrr'] * 2) + (live_pro_df['recent_wickets'] * 5) - (live_pro_df['recent_runs'] * 0.5)

# 6. Clean and Prepare for Re-Training
final_pro_df = live_pro_df[['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 
                            'wickets_left', 'runs_target', 'recent_runs', 'recent_wickets', 'pressure_index', 'match_won_by']]

# Define result: 1 if batting team won (match_won_by == batting_team)
final_pro_df['result'] = (final_pro_df['match_won_by'] == 'wickets').astype(int) # Simplification for testing
final_pro_df.dropna(inplace=True)

print(f"Advanced Features Integrated! New Shape: {final_pro_df.shape}")
display(final_pro_df.head())

Advanced Features Integrated! New Shape: (133903, 12)


,batting_team,bowling_team,city,runs_left,balls_left,wickets_left,runs_target,recent_runs,recent_wickets,pressure_index,match_won_by,result
124,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,222.0,125,10,223.0,1.0,0.0,20.811830,Kolkata Knight Riders,0
125,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,221.0,124,10,223.0,1.0,0.0,20.886924,Kolkata Knight Riders,0
126,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,221.0,124,10,223.0,1.0,0.0,20.886924,Kolkata Knight Riders,0
127,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,220.0,123,10,223.0,2.0,0.0,20.463240,Kolkata Knight Riders,0
128,Royal Challengers Bangalore,Kolkata Knight Riders,Bangalore,219.0,122,10,223.0,3.0,0.0,20.040807,Kolkata Knight Riders,0


In [17]:
from sklearn.ensemble import RandomForestClassifier

# 1. Clean up any leftover text columns before training
if 'match_won_by' in final_pro_df.columns:
    final_pro_df = final_pro_df.drop('match_won_by', axis=1)

# 2. Define Features (X) and Target (y)
X = final_pro_df.drop('result', axis=1)
y = final_pro_df['result']

# 3. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create the Advanced Pipeline
step1 = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

# We use max_depth=15 so it learns the complex pressure patterns without overfitting
step2 = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)

pipe_pro = Pipeline([
    ('step1', step1),
    ('step2', step2)
])

# 5. Train the Advanced Model
print("Training the Live Pro Engine... (This might take a few seconds)")
pipe_pro.fit(X_train, y_train)

# 6. Check the New Accuracy!
y_pred_pro = pipe_pro.predict(X_test)
pro_accuracy = accuracy_score(y_test, y_pred_pro)

print(f"🔥 Advanced Live Engine Accuracy: {pro_accuracy * 100:.2f}%")

Training the Live Pro Engine... (This might take a few seconds)
🔥 Advanced Live Engine Accuracy: 100.00%


In [18]:
# THE 100% ACCURACY FIX 

# 1. Start fresh with the proper merged data
ipl_df = pd.read_csv('../data/raw/IPL.csv')

# Re-calculate momentum
ipl_df['recent_runs'] = ipl_df.groupby(['match_id', 'innings'])['runs_batter'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())
ipl_df['recent_wickets'] = ipl_df.groupby(['match_id', 'innings'])['bowler_wicket'].transform(lambda x: x.rolling(window=18, min_periods=1).sum())

live_pro_df = ipl_df[ipl_df['innings'] == 2].copy()
live_pro_df['current_score'] = live_pro_df.groupby(['match_id', 'innings'])['runs_total'].cumsum()
live_pro_df['runs_left'] = live_pro_df['runs_target'] - live_pro_df['current_score']
live_pro_df['balls_left'] = 126 - (live_pro_df['over'] * 6 + live_pro_df['ball'])
live_pro_df['wickets_left'] = 10 - live_pro_df.groupby(['match_id', 'innings'])['bowler_wicket'].cumsum()
live_pro_df['rrr'] = (live_pro_df['runs_left'] * 6) / (live_pro_df['balls_left'] + 0.001)
live_pro_df['pressure_index'] = (live_pro_df['rrr'] * 2) + (live_pro_df['recent_wickets'] * 5) - (live_pro_df['recent_runs'] * 0.5)

# ---- THE FIX IS HERE ----
# Get the true winners from our matches df
true_winners = matches[['id', 'winner']].rename(columns={'id': 'match_id'})
live_pro_df = live_pro_df.merge(true_winners, on='match_id', how='inner')

# True Result: 1 if the chasing team (batting_team) matches the actual winner
live_pro_df['result'] = (live_pro_df['batting_team'] == live_pro_df['winner']).astype(int)

# Filter columns securely (NO LEAKY COLUMNS ALLOWED)
final_pro_fixed = live_pro_df[['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 
                            'wickets_left', 'runs_target', 'recent_runs', 'recent_wickets', 'pressure_index', 'result']]

# Clean up infinite values and NaNs
final_pro_fixed = final_pro_fixed[final_pro_fixed['balls_left'] > 0].dropna()

# 2. Train-Test Split & Train
X_real = final_pro_fixed.drop('result', axis=1)
y_real = final_pro_fixed['result']
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

print("Training the fixed Live Pro Engine...")
pipe_pro.fit(X_train, y_train)

# 3. Check the REAL Accuracy
real_accuracy = accuracy_score(y_test, pipe_pro.predict(X_test))
print(f"✅ REAL Advanced Live Engine Accuracy: {real_accuracy * 100:.2f}%")

Training the fixed Live Pro Engine...
✅ REAL Advanced Live Engine Accuracy: 92.13%


In [19]:
import joblib
import os

# 1. Ensure models directory exists
model_path = '../models/'
os.makedirs(model_path, exist_ok=True)

# 2. Save the full Pipeline (Encoder + Random Forest)
joblib.dump(pipe_pro, os.path.join(model_path, 'live_match_pipe.pkl'))

# 3. Save the exact list of columns
live_features = list(X_train.columns)
joblib.dump(live_features, os.path.join(model_path, 'live_match_features.pkl'))

print("✅ Phase 3 Complete: Live Engine pipeline successfully saved!")
print("Files saved: live_match_pipe.pkl, live_match_features.pkl")

✅ Phase 3 Complete: Live Engine pipeline successfully saved!
Files saved: live_match_pipe.pkl, live_match_features.pkl
